<a href="https://colab.research.google.com/github/KRSets07/python-course-project/blob/main/Week_5_Assignment_%7C_Supervised_Machine_Learning_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5 Assignment: Building My First Supervised Machine Learning Model

## Introduction

This project uses stope data from a mining project to predict whether a stope is stable or unstable. The dataset contains rock mass properties and stope measurements that may help explain stope stability.

The aim is to train two supervised machine learning models, Logistic Regression and Decision Tree, and compare their performance on a test dataset.

In [29]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [30]:
df = pd.read_excel("STOPE ID.xlsx", sheet_name="Stope Data")

df.head()

,Stope Name,Surface,Q',A,B,C,Top Length (m),Bottom Length (m),North Height (m),South Height (m),N',HR,ELOS (m)
0,TAU_115_STO_S_22,HW,2.0,1,0.4,4.3,34.420,34.090,32.601,30.540,3.4,8.1,1.736231
1,TAU_115_STO_S_22,FW,2.0,1,0.4,4.0,34.332,34.079,30.966,29.811,3.2,8.0,0.215763
2,NKW_115_STO_N_29,HW,2.0,1,0.3,4.7,30.080,30.126,31.609,32.008,2.8,7.8,1.466264
3,NKW_115_STO_N_29,FW,2.0,1,0.3,5.2,30.090,30.017,32.031,31.173,3.1,7.6,0.225319
4,TAU_115_STO_S_17,HW,2.0,1,0.3,3.9,34.024,34.258,36.381,32.733,2.3,8.4,1.329881


In [31]:
# Create the target variable using the correct column name "ELOS (m)"
df["Stability"] = (df["ELOS (m)"] >= 0.6).astype(int)

# 0 = Stable
# 1 = Unstable

# Select features using the exact column names present in the Excel sheet
feature_columns = [
    "Q'",
    "A",
    "B",
    "C",
    "Top Length (m)",
    "Bottom Length (m)",
    "North Height (m)",
    "South Height (m)",
    "N'",
    "HR",
    "Surface"
]

X = df[feature_columns].copy()
y = df["Stability"]

print("Features:")
print(X.head())

print("\nTarget distribution:")
print(y.value_counts())

Features:
    Q'  A    B    C  Top Length (m)  Bottom Length (m)  North Height (m)  \
0  2.0  1  0.4  4.3          34.420             34.090            32.601   
1  2.0  1  0.4  4.0          34.332             34.079            30.966   
2  2.0  1  0.3  4.7          30.080             30.126            31.609   
3  2.0  1  0.3  5.2          30.090             30.017            32.031   
4  2.0  1  0.3  3.9          34.024             34.258            36.381   

   South Height (m)   N'   HR Surface  
0            30.540  3.4  8.1      HW  
1            29.811  3.2  8.0      FW  
2            32.008  2.8  7.8      HW  
3            31.173  3.1  7.6      FW  
4            32.733  2.3  8.4      HW  

Target distribution:
Stability
0    202
1     92
Name: count, dtype: int64


## Problem Type

This is a classification problem because the target variable has two categories: stable and unstable. A regression model would be more suitable if the aim were to predict the actual ELOS value in metres.

The features include rock mass properties, stability parameters, stope dimensions and surface type. These variables are used to predict the stability class.

In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 235
Testing samples: 59


In [33]:
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [34]:
# Model 1: Logistic Regression
model1 = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model1.fit(X_train, y_train)

# Model 2: Decision Tree
model2 = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ))
])

model2.fit(X_train, y_train)

print("Both models have been trained.")

Both models have been trained.


In [35]:
y_pred_lr = model1.predict(X_test)
y_pred_dt = model2.predict(X_test)

print("Logistic Regression predictions:")
print(y_pred_lr[:10])

print("\nDecision Tree predictions:")
print(y_pred_dt[:10])

Logistic Regression predictions:
[0 0 0 0 0 0 1 0 1 0]

Decision Tree predictions:
[1 0 0 0 0 0 1 0 1 0]


In [36]:
def evaluate_model(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "F1-score": f1_score(
            y_true, y_pred, zero_division=0
        )
    }

results = pd.DataFrame([
    evaluate_model("Logistic Regression", y_test, y_pred_lr),
    evaluate_model("Decision Tree", y_test, y_pred_dt)
])

print(results.round(3).to_string(index=False))

              Model  Accuracy  Precision  Recall  F1-score
Logistic Regression     0.746      0.636   0.389     0.483
      Decision Tree     0.763      0.643   0.500     0.562


In [37]:
print("Logistic Regression Report")
print(classification_report(
    y_test, y_pred_lr,
    target_names=["Stable", "Unstable"],
    zero_division=0
))

print("Decision Tree Report")
print(classification_report(
    y_test, y_pred_dt,
    target_names=["Stable", "Unstable"],
    zero_division=0
))

Logistic Regression Report
              precision    recall  f1-score   support

      Stable       0.77      0.90      0.83        41
    Unstable       0.64      0.39      0.48        18

    accuracy                           0.75        59
   macro avg       0.70      0.65      0.66        59
weighted avg       0.73      0.75      0.73        59

Decision Tree Report
              precision    recall  f1-score   support

      Stable       0.80      0.88      0.84        41
    Unstable       0.64      0.50      0.56        18

    accuracy                           0.76        59
   macro avg       0.72      0.69      0.70        59
weighted avg       0.75      0.76      0.75        59



In [38]:
best_model = results.loc[
    results["F1-score"].idxmax()
]

print("Model with the highest F1-score:")
print(best_model["Model"])

print("\nIts F1-score:", round(best_model["F1-score"], 3))

Model with the highest F1-score:
Decision Tree

Its F1-score: 0.562


## Results and Discussion

The two models were evaluated using accuracy, precision, recall and F1-score. Logistic Regression achieved an accuracy of 74.6%, while the Decision Tree achieved 76.3%.

The Decision Tree performed better overall, with a higher F1-score of 56.2% for unstable stopes, compared with 48.3% for Logistic Regression. It also achieved a higher recall of 50%, meaning it identified half of the actual unstable stopes in the test dataset. Logistic Regression identified 38.9% of the unstable stopes.

Both models performed better at identifying stable stopes than unstable stopes. This suggests that the models have difficulty identifying some unstable cases. Although the Decision Tree performed better on this test dataset, further testing would be needed to determine whether its performance remains consistent on other data.

In conclusion, the Decision Tree was the better-performing model in this experiment. However, neither model identified all unstable stopes, so their predictions would need further validation before being used to support mining decisions.